# Watching a SuperPEEC solve live

This notebook launches a solve in a **separate process** and watches it
through the status API — the same mechanism a GUI or any other tool
would use. It shows a live progress line, a Z(f) plot that fills in as
frequency points complete, and a post-mortem of where the time went.

Reference: `docs/status_api.md` (schema 1). Run this notebook from the
`examples/` directory (Jupyter's default when you open it here).
Needs `matplotlib` and `tqdm` (both plain pip installs).


In [1]:
# -- launch: any SuperPEEC entry point works; the CLI is the simplest.
# The env var SPPEEC_STATUS makes the solver write an atomically
# updated JSON status file; SPPEEC_STATUS_EVENTS adds a JSONL
# transition log. Neither changes any converged number.
#
# The solver runs under THIS kernel's Python (sys.executable), so the
# kernel must live in an environment with the solver's dependencies
# -- if the launch fails instantly, that is the first thing to check
# (cell 3 will show you the recorded error either way).
import os
import subprocess
import sys
import tempfile

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))
import sppeec_status

# One FIXED workdir: re-running this cell reuses the same paths
# instead of littering the temp dir with one directory per run. The
# status file must be truncated (else cell 3 could read the PREVIOUS
# run's terminal state before the new solver's first write), and the
# event log appends, so truncating keeps one run's history per file.
workdir = os.path.join(tempfile.gettempdir(), 'sppeec_monitor')
os.makedirs(workdir, exist_ok=True)
STATUS = os.path.join(workdir, 'status.json')
EVENTS = os.path.join(workdir, 'events.jsonl')
LOG = os.path.join(workdir, 'solver.log')
if 'proc' in globals() and proc.poll() is None:
    raise RuntimeError('previous solve (pid %d) is still running -- '
                       'proc.terminate() first to restart' % proc.pid)
for p in (STATUS, EVENTS, LOG):
    open(p, 'w').close()

proc = subprocess.Popen(
    [sys.executable, os.path.join(ROOT, 'src', 'sppeec_cli.py'),
     os.path.join(ROOT, 'examples', 'module3wire.toml')],
    env=dict(os.environ, SPPEEC_STATUS=STATUS,
             SPPEEC_STATUS_EVENTS=EVENTS),
    cwd=ROOT, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
print('launching with', sys.executable)
print('solver pid', proc.pid, '-> status file', STATUS)


launching with /usr/bin/python3
solver pid 2613670 -> status file /tmp/sppeec_monitor/status.json


In [2]:
# -- live monitor: a persistent tqdm bar tracks overall percent (with
# the current task and frequency as its postfix), and the partial Z(f)
# sweep redraws in place -- via a display handle, only when a new
# point lands -- so nothing flickers. sppeec_status.read() adds
# _alive/_stale_s, so a crashed writer is detected rather than waited
# on forever.
import time

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

bar = tqdm(total=100.0, desc='overall', unit='%',
           bar_format='{l_bar}{bar}| {n:.0f}/{total:.0f}% {postfix}')
plot = None
shown = 0
def tail(path, n=15):
    try:
        return '\n'.join(open(path).read().splitlines()[-n:])
    except OSError:
        return '(no solver output captured)'

while True:
    try:
        d = sppeec_status.read(STATUS)
    except (FileNotFoundError, ValueError):
        # no (complete) status document yet -- fine while the solver
        # is starting up, fatal if it already died
        if proc.poll() is not None:
            bar.close()
            print('solver exited (rc=%d) before reporting any '
                  'status; log tail:' % proc.returncode)
            print(tail(LOG))
            break
        time.sleep(0.3)
        continue
    ov = d['overall']['pct']
    if ov is not None:
        bar.n = min(float(ov), 100.0)
    post = {'task': d['task'].get('current') or '-'}
    if d['sweep']['current_freq'] is not None:
        post['f'] = '%g (%d/%s)' % (d['sweep']['current_freq'],
                                    d['sweep']['index'] + 1,
                                    d['sweep']['n'])
    if d['task'].get('pct') is not None:
        post['task%'] = '%.0f' % d['task']['pct']
    bar.set_postfix(post, refresh=True)
    rows = d['sweep']['results']
    if len(rows) > shown and all('R' in r for r in rows):
        shown = len(rows)
        fig, (axr, axl) = plt.subplots(1, 2, figsize=(9, 3))
        fs = [r['f'] for r in rows]
        axr.loglog(fs, [r['R'] for r in rows], 'o-')
        axr.set_xlabel('f [Hz]'); axr.set_ylabel('R [Ohm]')
        if all(r.get('L') for r in rows):
            axl.semilogx(fs, [r['L'] * 1e9 for r in rows], 'o-')
            axl.set_xlabel('f [Hz]'); axl.set_ylabel('L [nH]')
        fig.suptitle('Z(f), filling in live (%d/%s points)'
                     % (len(rows), d['sweep']['n']))
        fig.tight_layout()
        if plot is None:
            plot = display(fig, display_id=True)
        else:
            plot.update(fig)
        plt.close(fig)
    if d['state'] != 'running':
        if d['state'] == 'done':
            bar.n = 100.0
        bar.refresh()
        bar.close()
        print('final state:', d['state'])
        if d.get('error'):
            print('recorded error:', d['error'])
        if d['state'] != 'done':
            print('solver log tail:')
            print(tail(LOG))
        break
    if not d['_alive']:
        bar.close()
        print('writer process is gone without finishing; log tail:')
        print(tail(LOG))
        break
    time.sleep(0.5)
rc = proc.wait()
print('solver exit code:', rc)


overall:   0%|          | 0/100% 

final state: failed
recorded error: ModuleNotFoundError("No module named 'pyfftw'")
solver log tail:
  File "/home/tims/Documents/sppeec/src/sppeec_cli.py", line 206, in _run
    M = prob.tree(m)
  File "/home/tims/Documents/sppeec/src/sppeec_input.py", line 819, in tree
    return m.build_tree(leaf, levels)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/home/tims/Documents/sppeec/src/voxmodel.py", line 571, in build_tree
    import multipole as mp
  File "/home/tims/Documents/sppeec/src/multipole.py", line 22, in <module>
    from multipole_common import *  # noqa: F401,F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/tims/Documents/sppeec/src/multipole_common.py", line 22, in <module>
    import toeplitz as tp
  File "/home/tims/Documents/sppeec/src/toeplitz.py", line 10, in <module>
    import pyfftw
ModuleNotFoundError: No module named 'pyfftw'
solver exit code: 1


In [ ]:
# -- post-mortem: the JSONL event log is append-only history. Summing
# task_end durations by task name says where the wall clock went.
import collections
import json

spent = collections.Counter()
for line in open(EVENTS):
    e = json.loads(line)
    if e['ev'] == 'task_end':
        spent[e['task']] += e['dur_s']

names = [n for n, _ in spent.most_common()]
vals = [spent[n] for n in names]
fig, ax = plt.subplots(figsize=(7, 0.4 * len(names) + 1))
ax.barh(range(len(names)), vals)
ax.set_yticks(range(len(names)), names)
ax.invert_yaxis()
ax.set_xlabel('wall seconds (summed over the run)')
ax.set_title('where the time went')
fig.tight_layout()
plt.show()


## Other ways to consume the same data

* **Terminal**: `python src/sppeec_status.py /path/status.json`
* **CLI live line**: `python src/sppeec_cli.py input.toml --status`
* **Same process** (a script driving the sweeper directly):

```python
import sppeec_status
sppeec_status.enable(callback=lambda d: my_widget.update(d))
```

The status file is written atomically and throttled (~4 Hz); poll it
from anything that can read a file. Schema and guarantees:
`docs/status_api.md`.
